# Phase 2 — Architecture 4: LegalBlock-GQA NER

In [2]:
%pip -q install "tokenizers>=0.20,<0.24" cloudpickle

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os, sys, json, math, time, random, hashlib, bisect, socket, subprocess, zipfile, gc, shutil
from pathlib import Path
from collections import Counter
from contextlib import nullcontext
from datetime import timedelta
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
from torch.utils.data import Dataset, DataLoader, DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP
import torch.distributed as dist
import cloudpickle
from tokenizers import Tokenizer

BASE = '/kaggle/input/datasets/raaz74/a4-p2'
DATA_DIR = '/kaggle/input/datasets/raaz74/a4-p2/Phase_2_LNER'
OUT = Path('/kaggle/working/phase2_arch4'); OUT.mkdir(parents=True, exist_ok=True)
BASE = Path(BASE)
DATA_DIR = Path(DATA_DIR)
C = json.loads((BASE/'config.json').read_text())
TOK = Tokenizer.from_file(str(BASE/'tokenizer.json'))
LABELS = (DATA_DIR/'labels.txt').read_text().split()
L2I = {x:i for i,x in enumerate(LABELS)}
TAGS = ['O'] + [p+'-'+l for l in LABELS for p in ['B','I']]
R = dict(seq=768, stride=512, micro=1, accum=4, epochs=12, stop_epoch=3,
         lr=2e-5, head_lr=2e-4, wd=.01, warmup=.1, patience=3, seed=781,
         dropout=.1, boundary_weight=.25, balanced=False, causal=False,
         method='full', rank=8, alpha=16, delta=16, teacher=None, kd=.5, temperature=2.,
         student=False, base=str(BASE), out=str(OUT), prepared=str(OUT/'prepared.pkl'))
for filename in ['config.json','tokenizer.json']:
    shutil.copyfile(BASE/filename, OUT/filename)
print('Inputs:', BASE, '|', DATA_DIR)

Inputs: /kaggle/input/datasets/raaz74/a4-p2 | /kaggle/input/datasets/raaz74/a4-p2/Phase_2_LNER


## 1. Inspect the official folds and freeze a document split

In [3]:
texts, gold, folds = {}, {}, {}
for f in [1,2]:
    t = json.loads((DATA_DIR/f'fold{f}-text.json').read_text())
    g = json.loads((DATA_DIR/f'fold{f}-gold.json').read_text())
    texts.update(t); gold.update(g); folds.update({k:f for k in t})
test = json.loads((DATA_DIR/'fold3-text.json').read_text())
counts = Counter(e[2] for rows in gold.values() for e in rows)
ids = sorted(texts)
entity_counts = np.array([[sum(e[2]==l for e in gold[k]) for l in LABELS] for k in ids])
rng = np.random.default_rng(R['seed']); best_cost = float('inf')
for _ in range(2000):
    v = sorted(sum([rng.choice([k for k in ids if folds[k]==f],7,replace=False).tolist() for f in [1,2]],[]))
    mask = np.array([k in v for k in ids]); vc = entity_counts[mask].sum(0)
    if (vc == 0).any() or (entity_counts[~mask].sum(0) == 0).any():
        continue
    cost = np.mean((vc / entity_counts.sum(0) - .2)**2)
    if cost < best_cost:
        best_cost, valid_ids = cost, v
train_ids = [k for k in ids if k not in valid_ids]
split = dict(train=train_ids, valid=valid_ids, seed=R['seed'])
(OUT/'split.json').write_text(json.dumps(split,indent=2))
print(f'{len(texts)} labelled documents | {sum(counts.values()):,} entities | {len(test)} test documents')
print('Train / validation:',len(train_ids),len(valid_ids))
print('Entities by class:',dict(counts))

70 labelled documents | 8,393 entities | 35 test documents
Train / validation: 56 14
Entities by class: {'RESP': 381, 'DATE': 1575, 'COURT': 1013, 'WIT': 187, 'APP': 407, 'STAT': 1260, 'A.COUNSEL': 195, 'PREC': 1183, 'JUDGE': 245, 'R.COUNSEL': 184, 'CASENO': 760, 'AUTH': 1003}


## 2. Exact character alignment and window ownership

In [4]:
def encode_doc(text, entities):
    enc = TOK.encode(text, add_special_tokens=False)
    off = np.array(enc.offsets, dtype=np.int32).reshape(-1,2)
    starts, ends = off[:,0].tolist(), off[:,1].tolist()
    y = np.zeros(len(off),dtype=np.int64)
    ds = np.full(len(off),-100,dtype=np.int64); de = ds.copy()
    spans = []
    for s,e,l,*_ in entities:
        a = bisect.bisect_right(ends,s); b = bisect.bisect_left(starts,e)-1
        if not (0 <= a <= b < len(off) and starts[a] <= s < ends[a] and starts[b] < e <= ends[b]):
            raise ValueError('Gold endpoint is not contained in a BPE token.')
        if y[a:b+1].any():
            raise ValueError('Two entities share a token; use a span head for this data.')
        ds[a], de[b] = s-starts[a], ends[b]-e
        if max(ds[a],de[b]) >= R['delta']:
            raise ValueError('Increase boundary head size and repeat the alignment audit.')
        y[a] = 1+2*L2I[l]; y[a+1:b+1] = 2+2*L2I[l]
        spans.append((a,b,L2I[l],s,e))
    return dict(text=text, ids=np.array(enc.ids,dtype=np.int64), off=off, y=y, ds=ds, de=de, spans=spans)

def windows(n, seq, stride):
    if n == 0: return []
    starts = list(range(0,max(1,n-seq+1),stride))
    if starts[-1]+seq < n:
        starts.append(n-seq)
    ends = [min(n,s+seq) for s in starts]
    cuts = [0] + [(ends[i-1]+starts[i])//2 for i in range(1,len(starts))] + [n]
    return [(s,e,cuts[i],cuts[i+1]) for i,(s,e) in enumerate(zip(starts,ends))]

records = {k:encode_doc(t,gold.get(k,[])) for k,t in {**texts,**test}.items()}
if set(texts) & set(test):
    raise ValueError('Training and test IDs overlap.')
trim_exact = 0; recovered = {}; max_span = 0
for k in ids:
    r = records[k]; recovered[k] = []
    owner = np.zeros(len(r['ids']),dtype=np.int32)
    for s,e,a,b in windows(len(owner),R['seq'],R['stride']):
        owner[a:b] += 1
        for x,z,_,_,_ in r['spans']:
            if x < b and z >= a and not (s <= x <= z < e):
                raise ValueError('Increase overlap: a supervised entity is cut by its context window.')
    if not np.all(owner==1): raise ValueError('Window ownership is not exact.')
    for a,b,l,s,e in r['spans']:
        ps = int(r['off'][a,0]+r['ds'][a]); pe = int(r['off'][b,1]-r['de'][b])
        recovered[k].append([ps,pe,LABELS[l]])
        q,w = int(r['off'][a,0]),int(r['off'][b,1])
        while q<w and r['text'][q].isspace(): q+=1
        while w>q and r['text'][w-1].isspace(): w-=1
        trim_exact += (q,w)==(s,e); max_span=max(max_span,b-a+1)
    if {tuple(x) for x in recovered[k]} != {tuple(x[:3]) for x in gold[k]}:
        raise ValueError('Gold span round-trip failed.')
print(f'Exact round-trip: {sum(map(len,recovered.values())):,}/{sum(counts.values()):,}; trimmed: {trim_exact:,}; max entity: {max_span} tokens')
manifest = dict(tokenizer_sha256=hashlib.sha256((BASE/'tokenizer.json').read_bytes()).hexdigest(),
                data_sha256={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in DATA_DIR.glob('*') if p.is_file()},
                split=split, counts=dict(counts), trimmed_exact=trim_exact, max_entity_tokens=max_span)
(OUT/'manifest.json').write_text(json.dumps(manifest,indent=2))
with (OUT/'prepared.pkl').open('wb') as f:
    cloudpickle.dump(dict(records=records,gold=gold,split=split,labels=LABELS,config=C,manifest=manifest),f)

Exact round-trip: 8,393/8,393; trimmed: 8,241; max entity: 53 tokens


## 3. Checkpoint-compatible backbone and NER head

In [5]:
class RMSNorm(nn.Module):
    def __init__(self,d):
        super().__init__(); self.weight = nn.Parameter(torch.ones(d))
    def forward(self,x):
        return (x.float()*torch.rsqrt(x.float().square().mean(-1,keepdim=True)+1e-6)).to(x.dtype)*self.weight

class Attention(nn.Module):
    def __init__(self,c,global_attn,causal):
        super().__init__()
        self.h,self.kv,self.d = c['heads'],c['kv_heads'],c['dim']//c['heads']
        self.window,self.global_attn,self.causal = c['window'],global_attn,causal
        self.qkv = nn.Linear(c['dim'],(self.h+2*self.kv)*self.d,bias=False)
        self.out = nn.Linear(c['dim'],c['dim'],bias=False)
        self.register_buffer('freq',1/(10000**(torch.arange(0,self.d,2).float()/self.d)),persistent=False)
    def rope(self,x):
        angle=(torch.arange(x.size(2),device=x.device).float()[:,None]*self.freq).to(x.dtype)
        a,b=x[...,0::2],x[...,1::2]
        return torch.stack((a*angle.cos()-b*angle.sin(),a*angle.sin()+b*angle.cos()),-1).flatten(-2)
    def forward(self,x,valid):
        b,t,_=x.shape
        q,k,v=self.qkv(x).split([self.h*self.d,self.kv*self.d,self.kv*self.d],-1)
        q=self.rope(q.view(b,t,self.h,self.d).transpose(1,2))
        k=self.rope(k.view(b,t,self.kv,self.d).transpose(1,2))
        v=v.view(b,t,self.kv,self.d).transpose(1,2)
        k,v=[z.repeat_interleave(self.h//self.kv,1) for z in (k,v)]
        if self.global_attn:
            mask=valid[:,None,None,:]
            if self.causal:
                mask=mask & torch.ones(t,t,device=x.device,dtype=torch.bool).tril()[None,None]
            y=F.scaled_dot_product_attention(q,k,v,attn_mask=mask)
        else:
            w=self.window; pad=(-t)%w
            q,k,v=[F.pad(z,(0,0,0,pad)).reshape(b,self.h,-1,w,self.d)
                   .permute(0,2,1,3,4).reshape(-1,self.h,w,self.d) for z in (q,k,v)]
            mask=F.pad(valid,(0,pad)).reshape(-1,w)[:,None,None,:]
            if self.causal:
                mask=mask & torch.ones(w,w,device=x.device,dtype=torch.bool).tril()[None,None]
            y=F.scaled_dot_product_attention(q,k,v,attn_mask=mask)
            y=y.reshape(b,-1,self.h,w,self.d).permute(0,2,1,3,4).reshape(b,self.h,-1,self.d)[:,:,:t]
        return self.out(y.transpose(1,2).reshape(b,t,-1))*valid[:,:,None]

class Block(nn.Module):
    def __init__(self,c,i,causal):
        super().__init__(); self.n1,self.n2=RMSNorm(c['dim']),RMSNorm(c['dim'])
        self.attn=Attention(c,(i+1)%c['global_every']==0,causal)
        self.up=nn.Linear(c['dim'],2*c['ff'],bias=False)
        self.down=nn.Linear(c['ff'],c['dim'],bias=False)
    def forward(self,x,valid):
        x=x+self.attn(self.n1(x),valid); a,b=self.up(self.n2(x)).chunk(2,-1)
        return x+self.down(F.silu(a)*b)

class NER(nn.Module):
    def __init__(self,c,r):
        super().__init__(); self.c,self.r=c,r
        self.source_layers = ([i for i in range(c['layers']) if i%4 in [0,3]] if r['student'] else list(range(c['layers'])))
        self.emb=nn.Embedding(c['vocab'],c['dim'])
        self.blocks=nn.ModuleList([Block(c,i,r['causal']) for i in self.source_layers])
        self.norm=RMSNorm(c['dim']); self.drop=nn.Dropout(r['dropout'])
        self.tag=nn.Linear(c['dim'],len(TAGS))
        self.start=nn.Linear(c['dim'],r['delta']); self.end=nn.Linear(c['dim'],r['delta'])
    def forward(self,ids,valid):
        x=self.emb(ids)
        for block in self.blocks:
            x=checkpoint(block,x,valid,use_reentrant=False) if self.training else block(x,valid)
        x=self.drop(self.norm(x))
        return self.tag(x),self.start(x),self.end(x)

class LoRMA(nn.Module):
    def __init__(self,base,rank,alpha):
        super().__init__(); self.base=base; self.scale=alpha/rank
        self.a=nn.Linear(base.out_features,rank,bias=False)
        self.b=nn.Linear(rank,base.out_features,bias=False)
        nn.init.kaiming_uniform_(self.a.weight,a=math.sqrt(5)); nn.init.zeros_(self.b.weight)
    def forward(self,x):
        y=self.base(x)
        return y+self.scale*self.b(self.a(y))

def adapt(model,r):
    if r['method']=='full': return
    model.requires_grad_(False)
    if r['method']=='partial':
        for block in model.blocks[-8:]: block.requires_grad_(True)
        model.norm.requires_grad_(True)
    elif r['method']=='lorma':
        for block in model.blocks:
            for parent,name in [(block.attn,'qkv'),(block.attn,'out'),(block,'up'),(block,'down')]:
                setattr(parent,name,LoRMA(getattr(parent,name),r['rank'],r['alpha']))
    else:
        raise ValueError('Use full, partial, or lorma.')
    for head in [model.tag,model.start,model.end]: head.requires_grad_(True)

def make_model(c,r,phase1=True):
    model=NER(c,r)
    if phase1:
        ck=torch.load(Path(r['base'])/'best.pt',map_location='cpu',weights_only=False)
        weights=ck['model']
        if r['student']:
            remap={old:new for new,old in enumerate(model.source_layers)}; state={}
            for name,value in weights.items():
                if name.startswith('blocks.'):
                    parts=name.split('.'); old=int(parts[1])
                    if old not in remap: continue
                    parts[1]=str(remap[old]); name='.'.join(parts)
                state[name]=value
            weights=state
        missing,extra=model.load_state_dict(weights,strict=False)
        expected={f'{h}.{p}' for h in ['tag','start','end'] for p in ['weight','bias']}
        if set(missing)!=expected or extra:
            raise ValueError(f'Phase 1 checkpoint mismatch: {missing}, {extra}')
        del ck,weights
    adapt(model,r)
    return model

## Results and interpretation

In [6]:
class Chunks(Dataset):
    def __init__(self,records,ids,r):
        self.records,self.rows=records,[]
        for k in ids:
            self.rows.extend((k,*w) for w in windows(len(records[k]['ids']),r['seq'],r['stride']))
    def __len__(self): return len(self.rows)
    def __getitem__(self,i):
        k,s,e,a,b=self.rows[i]; r=self.records[k]
        target=[]
        for name in ['y','ds','de']:
            v=r[name][s:e].copy(); v[:a-s]=-100; v[b-s:]=-100; target.append(v)
        return r['ids'][s:e],*target

def collate(rows):
    length=(max(len(r[0]) for r in rows)+31)//32*32
    x=torch.full((len(rows),length),C['pad'],dtype=torch.long)
    mask=torch.zeros_like(x,dtype=torch.bool)
    ys=[torch.full_like(x,-100) for _ in range(3)]
    for i,row in enumerate(rows):
        n=len(row[0]); x[i,:n]=torch.from_numpy(row[0]); mask[i,:n]=True
        for dst,src in zip(ys,row[1:]): dst[i,:n]=torch.from_numpy(src)
    return x,mask,*ys

def loss_fn(output,batch,weights,r,teacher=None):
    tag,st,en=[x.float() for x in output]; y,ds,de=batch[2:]
    main=F.cross_entropy(tag.flatten(0,1),y.flatten(),weight=weights,ignore_index=-100)
    boundary=tag.sum()*0
    for logits,target in [(st,ds),(en,de)]:
        active=target!=-100
        boundary=boundary+F.cross_entropy(logits[active],target[active]) if active.any() else boundary+logits.sum()*0
    loss=main+r['boundary_weight']*boundary
    if teacher is not None:
        active=y!=-100; temp=r['temperature']; soft=F.softmax(teacher[0][active].float()/temp,-1)
        kl=F.kl_div(F.log_softmax(tag[active]/temp,-1),soft,reduction='none').sum(-1)*temp**2
        entities=y[active]!=0; parts=[kl[flag].mean() for flag in [entities,~entities] if flag.any()]
        loss=(1-r['kd'])*loss+r['kd']*torch.stack(parts).mean()
    return loss

def class_weights(gold,train_ids,balanced,device):
    w=torch.ones(len(TAGS),device=device)
    if balanced:
        count=Counter(e[2] for k in train_ids for e in gold[k])
        values=np.array([count[l] for l in LABELS],dtype=float)
        values=np.clip(np.sqrt(np.median(values)/np.maximum(values,1)),.7,2.5)
        w[1:]=torch.tensor(np.repeat(values,2),device=device,dtype=w.dtype)
    return w

In [7]:
def viterbi(logits,bias=0.):
    z=np.array(logits,dtype=np.float32,copy=True); z[:,1:]+=bias
    n,k=z.shape
    if n == 0: return np.empty(0,dtype=np.int16)
    trans=np.zeros((k,k),dtype=np.float32)
    for j in range(2,k,2):
        trans[:,j]=-1e4; trans[j-1:j+1,j]=0
    score=z[0].copy(); score[2::2]=-1e4
    back=np.empty((n,k),dtype=np.int16)
    for i in range(1,n):
        candidates=score[:,None]+trans
        back[i]=candidates.argmax(0); score=candidates[back[i],np.arange(k)]+z[i]
    path=np.empty(n,dtype=np.int16); path[-1]=score.argmax()
    for i in range(n-1,0,-1): path[i-1]=back[i,path[i]]
    return path

def decode(r,outputs,refine=True,bias=0.):
    tag,st,en=outputs; path=viterbi(tag,bias); spans=[]; i=0
    while i<len(path):
        if path[i]==0: i+=1; continue
        label=(int(path[i])-1)//2; j=i
        while j+1<len(path) and path[j+1]==2+2*label: j+=1
        s,e=int(r['off'][i,0]),int(r['off'][j,1])
        if refine:
            ns=min(st.shape[-1],int(r['off'][i,1]-s))
            ne=min(en.shape[-1],int(e-r['off'][j,0]))
            if i==j:
                scores=st[i,:ns,None]+en[j,None,:ne]
                scores=np.where(np.arange(ns)[:,None]+np.arange(ne)[None,:]<e-s,scores,-np.inf)
                a,b=np.unravel_index(scores.argmax(),scores.shape)
            else:
                a,b=st[i,:ns].argmax(),en[j,:ne].argmax()
            s+=int(a); e-=int(b)
        else:
            while s<e and r['text'][s].isspace(): s+=1
            while e>s and r['text'][e-1].isspace(): e-=1
        if s<e: spans.append([s,e,LABELS[label]])
        i=j+1
    return sorted(set(map(tuple,spans)))

def score_ner(pred,gold,ids):
    counts=np.zeros((len(LABELS),3),dtype=np.int64)
    for k in ids:
        p=set(map(tuple,pred[k])); g={tuple(e[:3]) for e in gold[k]}
        for i,l in enumerate(LABELS):
            a={x for x in p if x[2]==l}; b={x for x in g if x[2]==l}
            counts[i]+=[len(a&b),len(a-b),len(b-a)]
    tp,fp,fn=counts.T
    precision=tp/np.maximum(1,tp+fp); recall=tp/np.maximum(1,tp+fn)
    f1=2*tp/np.maximum(1,2*tp+fp+fn)
    return dict(macro_f1=float(f1.mean()),macro_precision=float(precision.mean()),macro_recall=float(recall.mean()),
                micro_f1=float(2*tp.sum()/max(1,2*tp.sum()+fp.sum()+fn.sum())),
                per_class={l:dict(tp=int(tp[i]),fp=int(fp[i]),fn=int(fn[i]),f1=float(f1[i]),precision=float(precision[i]),recall=float(recall[i]),support=int(tp[i]+fn[i])) for i,l in enumerate(LABELS)})

@torch.inference_mode()
def infer_doc(model,r,run,device):
    n=len(r['ids']); dims=[len(TAGS),run['delta'],run['delta']]
    sums=[np.zeros((n,d),dtype=np.float32) for d in dims]; den=np.zeros(n,dtype=np.float32)
    for s,e,_,_ in windows(n,run['seq'],run['stride']):
        x=torch.tensor(r['ids'][s:e],device=device)[None]; valid=torch.ones_like(x,dtype=torch.bool)
        with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=device.type=='cuda'):
            outputs=model(x,valid)
        weight=np.minimum(np.arange(e-s)+1,np.arange(e-s,0,-1)).astype(np.float32)
        weight=np.minimum(weight,128)/128
        for total,z in zip(sums,outputs): total[s:e]+=z[0].float().cpu().numpy()*weight[:,None]
        den[s:e]+=weight
    return [z/den[:,None] for z in sums]

@torch.inference_mode()
def validate(model,records,gold,ids,run,device):
    model.eval(); pred={k:decode(records[k],infer_doc(model,records[k],run,device)) for k in ids}
    return score_ner(pred,gold,ids),pred

## 4. Component gates (CPU)

In [8]:
def component_checks():
    torch.manual_seed(781); torch.set_num_threads(2)
    c=dict(vocab=64,dim=32,layers=4,heads=4,kv_heads=2,ff=64,window=8,global_every=4)
    r=dict(R,dropout=0.,causal=False,student=False,method='full')
    model=NER(c,r).eval(); x=torch.randint(1,64,(1,13)); mask=torch.ones_like(x,dtype=torch.bool)
    a=model(x,mask)
    padded=F.pad(x,(0,3)); pm=F.pad(mask,(0,3))
    b=model(padded,pm)
    for u,v in zip(a,b): torch.testing.assert_close(u,v[:,:13],atol=2e-5,rtol=2e-5)
    local=Attention(c,False,False).eval(); h=torch.randn(1,16,32); valid=torch.ones(1,16,dtype=torch.bool)
    before=local(h,valid); changed=h.clone(); changed[:,8:]+=5
    torch.testing.assert_close(before[:,:8],local(changed,valid)[:,:8])
    causal=Attention(c,True,True).eval(); changed=h.clone(); changed[:,7:]+=5
    torch.testing.assert_close(causal(h,valid)[:,:7],causal(changed,valid)[:,:7])
    bi=Attention(c,True,False).eval()
    assert not torch.allclose(bi(h,valid)[:,:7],bi(changed,valid)[:,:7])
    base=nn.Linear(32,48,bias=False); wrapper=LoRMA(base,4,8)
    torch.testing.assert_close(base(h),wrapper(h))
    wrapper(h).square().mean().backward(); assert wrapper.b.weight.grad.abs().sum()>0
    model.train(); sum(z.square().mean() for z in model(x,mask)).backward()
    assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters())
    z=np.zeros((3,len(TAGS)),np.float32); z[:,2]=20
    assert viterbi(z)[0]!=2
    g={'x':[[0,2,LABELS[0]]], 'y':[[1,3,LABELS[1]]]}
    perfect=score_ner({k:[v[0][:3]] for k,v in g.items()},g,['x','y'])
    assert abs(perfect['macro_f1']-2/12)<1e-9
    shifted=score_ner({'x':[(0,3,LABELS[0])],'y':[(1,3,LABELS[1])]},g,['x','y'])
    assert abs(shifted['macro_f1']-1/12)<1e-9
    print('Component checks passed.')
component_checks()
for key in ids:
    rec=records[key]; length=len(rec['ids'])
    z=np.full((length,len(TAGS)),-20.,np.float32); z[np.arange(length),rec['y']]=20
    endpoints=[]
    for name in ['ds','de']:
        target=np.maximum(rec[name],0); a=np.full((length,R['delta']),-20.,np.float32)
        a[np.arange(length),target]=20; endpoints.append(a)
    assert set(decode(rec,[z,*endpoints]))=={tuple(e[:3]) for e in gold[key]}
assert decode(records['1305957'],[np.empty((0,d)) for d in [len(TAGS),R['delta'],R['delta']]])==[]
print('Full decoder gold test: all 8,393 entities exact; empty test ID retained.')

Component checks passed.
Full decoder gold test: all 8,393 entities exact; empty test ID retained.


## 5. Kaggle overfit gate and notebook-only DDP launcher

In [9]:
def launch(fn,*args):
    payload=OUT/'worker.pkl'
    with payload.open('wb') as f: cloudpickle.dump((fn,args),f)
    with socket.socket() as s:
        s.bind(('127.0.0.1',0)); port=s.getsockname()[1]
    command="import cloudpickle,os; f=open(os.environ['WORKER_PAYLOAD'],'rb'); fn,args=cloudpickle.load(f); f.close(); fn(int(os.environ['RANK']),*args)"
    jobs=[]
    for rank in range(2):
        env=dict(os.environ,MASTER_ADDR='127.0.0.1',MASTER_PORT=str(port),RANK=str(rank),
                 WORKER_PAYLOAD=str(payload),OMP_NUM_THREADS='2',TOKENIZERS_PARALLELISM='false')
        jobs.append(subprocess.Popen([sys.executable,'-u','-c',command],env=env))
    while any(p.poll() is None for p in jobs):
        if any(p.poll() not in (None,0) for p in jobs):
            for p in jobs:
                if p.poll() is None: p.terminate()
            raise RuntimeError('GPU worker failed; see its traceback above.')
        time.sleep(.5)
    if any(p.returncode for p in jobs): raise RuntimeError('GPU worker failed.')


def smoke_test():
    device=torch.device('cuda:0'); torch.manual_seed(R['seed'])
    run=dict(R,causal=False); model=make_model(C,run).to(device)
    dataset=Chunks(records,train_ids,run)
    candidates=sorted(range(len(dataset)),key=lambda i:int((dataset[i][1]>0).sum()),reverse=True)
    chosen=[candidates[0]]
    chosen.append(next(i for i in candidates if dataset.rows[i][0]!=dataset.rows[chosen[0]][0]))
    batch=[x.to(device) for x in collate([dataset[i] for i in chosen])]
    opt=torch.optim.AdamW(model.parameters(),lr=8e-5,weight_decay=0.,fused=True)
    scaler=torch.amp.GradScaler('cuda',init_scale=1024)
    weights=torch.ones(len(TAGS),device=device); losses=[]
    model.train(); torch.cuda.reset_peak_memory_stats()
    for step in range(80):
        opt.zero_grad(set_to_none=True)
        with torch.autocast('cuda',dtype=torch.float16):
            output=model(*batch[:2]); loss=loss_fn(output,batch,weights,run)
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
        scaler.step(opt); scaler.update(); losses.append(float(loss.detach()))
    before,after=float(np.mean(losses[:5])),float(np.mean(losses[-5:]))
    peak=torch.cuda.max_memory_allocated()/2**30
    print(f'Overfit gate: loss {before:.4f} → {after:.4f} | peak {peak:.2f} GiB')
    del model,opt,scaler,batch,output,loss; gc.collect(); torch.cuda.empty_cache()
    if not np.isfinite(after) or after>=.5*before:
        raise RuntimeError('Overfit gate failed; inspect initialization/loss before a full run.')
    (OUT/'smoke.json').write_text(json.dumps(dict(initial_loss=before,final_loss=after,peak_gib=peak)))

smoke_test()

Overfit gate: loss 2.2331 → 0.0000 | peak 3.93 GiB


In [10]:
def train_worker(rank,run,resume=False):
    torch.set_num_threads(2); torch.cuda.set_device(rank)
    dist.init_process_group('nccl',rank=rank,world_size=2,timeout=timedelta(minutes=60))
    device=torch.device('cuda',rank); out=Path(run['out'])/run['name']; out.mkdir(exist_ok=True)
    with open(run['prepared'],'rb') as f: pack=cloudpickle.load(f)
    records,gold,c=pack['records'],pack['gold'],pack['config']
    tr,va=pack['split']['train'],pack['split']['valid']
    if run.get('refit'): tr=sorted(gold); va=[]
    torch.manual_seed(run['seed']); np.random.seed(run['seed']+rank); random.seed(run['seed']+rank)
    model=make_model(c,run,phase1=not resume).to(device)
    teacher=None
    if run['teacher']:
        tc=torch.load(run['teacher'],map_location='cpu',weights_only=False)
        if tc['manifest'] != pack['manifest']: raise ValueError('Teacher data/split differs from the student.')
        if tc['run'].get('refit') and not run.get('refit'):
            raise ValueError('A refitted teacher has seen validation; use a training-only teacher.')
        teacher=make_model(tc['config'],tc['run'],phase1=False)
        teacher.load_state_dict(tc['model']); teacher.to(device).eval().requires_grad_(False); del tc
    groups=[]
    for head in [False,True]:
        for decay in [False,True]:
            ps=[p for name,p in model.named_parameters() if p.requires_grad and
                name.startswith(('tag.','start.','end.'))==head and (p.ndim>=2)==decay]
            if ps: groups.append(dict(params=ps,lr=run['head_lr'] if head else run['lr'],
                                      initial_lr=run['head_lr'] if head else run['lr'],weight_decay=run['wd'] if decay else 0.))
    opt=torch.optim.AdamW(groups,betas=(.9,.999),eps=1e-8,fused=True)
    scaler=torch.amp.GradScaler('cuda',init_scale=1024)
    dataset=Chunks(records,tr,run)
    sampler=DistributedSampler(dataset,num_replicas=2,rank=rank,shuffle=True,seed=run['seed'])
    loader=DataLoader(dataset,batch_size=run['micro'],sampler=sampler,collate_fn=collate,
                      num_workers=0,pin_memory=True)
    weights=class_weights(gold,tr,run['balanced'],device)
    start_epoch,step,best,wait,best_epoch=0,0,-1.,0,0
    torch.manual_seed(run['seed']+rank)
    if resume:
        ck=torch.load(out/'last.pt',map_location='cpu',weights_only=False)
        if ck['manifest'] != pack['manifest']: raise ValueError('Resume data/tokenizer/split changed.')
        for key in ['seq','stride','epochs','causal','method','student','balanced','seed','teacher']:
            if ck['run'][key]!=run[key]: raise ValueError(f'Resume configuration changed: {key}')
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['optimizer']); scaler.load_state_dict(ck['scaler'])
        start_epoch,step,best,wait,best_epoch=ck['epoch'],ck['step'],ck['best'],ck['wait'],ck['best_epoch']
        torch.set_rng_state(ck['rng'][rank]['cpu']); torch.cuda.set_rng_state(ck['rng'][rank]['cuda'],device)
        np.random.set_state(ck['rng'][rank]['numpy']); random.setstate(ck['rng'][rank]['python']); del ck
    ddp=DDP(model,device_ids=[rank],broadcast_buffers=False,gradient_as_bucket_view=True)
    total=math.ceil(len(loader)/run['accum'])*run['epochs']; warm=max(1,int(total*run['warmup']))
    if rank==0:
        print(f"{run['name']}: {sum(p.numel() for p in model.parameters()):,} parameters | "
              f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable | {len(dataset)} windows",flush=True)
        (out/'run.json').write_text(json.dumps(run,indent=2))
    for epoch in range(start_epoch,run['stop_epoch']):
        if va and wait>=run['patience']: break
        sampler.set_epoch(epoch); ddp.train(); opt.zero_grad(set_to_none=True)
        torch.cuda.reset_peak_memory_stats(); started=time.monotonic(); running=torch.zeros((),device=device)
        for i,cpu_batch in enumerate(loader):
            batch=[x.to(device,non_blocking=True) for x in cpu_batch]
            size=min(run['accum'],len(loader)-(i//run['accum'])*run['accum'])
            sync=(i+1)%run['accum']==0 or i+1==len(loader)
            ratio=(step+1)/warm if step<warm else .1+.9*.5*(1+math.cos(math.pi*min(1.,(step-warm)/max(1,total-warm))))
            for group in opt.param_groups: group['lr']=group['initial_lr']*ratio
            with nullcontext() if sync else ddp.no_sync():
                with torch.autocast('cuda',dtype=torch.float16):
                    soft=None
                    if teacher is not None:
                        with torch.no_grad(): soft=teacher(*batch[:2])
                    outputs=ddp(*batch[:2]); loss=loss_fn(outputs,batch,weights,run,soft)
                scaler.scale(loss/size).backward()
            running+=loss.detach()
            if sync:
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
                old_scale=scaler.get_scale(); scaler.step(opt); scaler.update()
                if scaler.get_scale()>=old_scale: step+=1
                opt.zero_grad(set_to_none=True)
        model.eval(); torch.cuda.synchronize(); seconds=time.monotonic()-started
        peak=torch.tensor(torch.cuda.max_memory_allocated()/2**30,device=device)
        dist.all_reduce(peak,op=dist.ReduceOp.MAX); dist.all_reduce(running)
        local={k:decode(records[k],infer_doc(model,records[k],run,device)) for k in va[rank::2]}
        gathered=[None,None]; dist.all_gather_object(gathered,local)
        pred={k:v for part in gathered for k,v in part.items()}
        metrics=score_ner(pred,gold,va) if va else dict(macro_f1=0.)
        improved=metrics['macro_f1']>best or not va
        if improved: best=metrics['macro_f1']; best_epoch=epoch+1; wait=0
        else: wait+=1
        state=dict(cpu=torch.get_rng_state(),cuda=torch.cuda.get_rng_state(device),
                   numpy=np.random.get_state(),python=random.getstate())
        states=[None,None]; dist.all_gather_object(states,state)
        if rank==0:
            log=dict(epoch=epoch+1,step=step,loss=float(running)/(2*len(loader)),
                     seconds=seconds,peak_gib=float(peak),metrics=metrics)
            with (out/'history.jsonl').open('a' if resume or epoch else 'w') as f: f.write(json.dumps(log)+'\n')
            print(f"{run['name']} epoch {epoch+1} | loss {log['loss']:.4f} | macro-F1 {metrics['macro_f1']:.4f} | {seconds/60:.1f} min | {float(peak):.2f} GiB",flush=True)
            ck=dict(model=model.state_dict(),config=c,run=run,epoch=epoch+1,step=step,best=best,
                    best_epoch=best_epoch,wait=wait,metrics=metrics,manifest=pack['manifest'])
            if improved:
                torch.save(ck,out/'best.tmp'); os.replace(out/'best.tmp',out/'best.pt')
                (out/'valid_predictions.json').write_text(json.dumps(pred))
                (out/'best_metrics.json').write_text(json.dumps(dict(epoch=best_epoch,**metrics),indent=2))
            ck.update(optimizer=opt.state_dict(),scaler=scaler.state_dict(),rng=states)
            torch.save(ck,out/'last.tmp'); os.replace(out/'last.tmp',out/'last.pt')
        dist.barrier()
    dist.destroy_process_group()

## 6. Equal-budget pilots, then promote one

In [11]:
pilot_runs=[dict(R,name='full_causal',causal=True),dict(R,name='full_bidirectional',causal=False)]
for run in pilot_runs:
    launch(train_worker,run,False)
pilot_scores={r['name']:json.loads((OUT/r['name']/'best_metrics.json').read_text())['macro_f1'] for r in pilot_runs}
winner=max(pilot_runs,key=lambda r:pilot_scores[r['name']])
selected_run=dict(winner,stop_epoch=R['epochs'])
launch(train_worker,selected_run,True)
CANDIDATES=[str(OUT/r['name']/'best.pt') for r in pilot_runs]
print('Pilot scores:',pilot_scores,'| promoted:',winner['name'])

full_causal: 261,590,457 parameters | 261,590,457 trainable | 747 windows
full_causal epoch 1 | loss 1.8414 | macro-F1 0.2682 | 1.5 min | 5.44 GiB


/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


full_causal epoch 2 | loss 0.1087 | macro-F1 0.4399 | 1.5 min | 5.43 GiB
full_causal epoch 3 | loss 0.0444 | macro-F1 0.4769 | 1.5 min | 5.43 GiB
full_bidirectional: 261,590,457 parameters | 261,590,457 trainable | 747 windows
full_bidirectional epoch 1 | loss 1.5392 | macro-F1 0.1406 | 1.5 min | 5.44 GiB


/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


full_bidirectional epoch 2 | loss 0.1374 | macro-F1 0.3297 | 1.5 min | 5.43 GiB
full_bidirectional epoch 3 | loss 0.0591 | macro-F1 0.4054 | 1.5 min | 5.43 GiB
full_causal: 261,590,457 parameters | 261,590,457 trainable | 747 windows
full_causal epoch 4 | loss 0.0189 | macro-F1 0.4422 | 1.5 min | 5.43 GiB


/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


full_causal epoch 5 | loss 0.0084 | macro-F1 0.4704 | 1.5 min | 5.43 GiB
full_causal epoch 6 | loss 0.0039 | macro-F1 0.4880 | 1.5 min | 5.43 GiB
full_causal epoch 7 | loss 0.0017 | macro-F1 0.4715 | 1.5 min | 5.43 GiB
full_causal epoch 8 | loss 0.0009 | macro-F1 0.4745 | 1.5 min | 5.43 GiB
full_causal epoch 9 | loss 0.0007 | macro-F1 0.4731 | 1.5 min | 5.43 GiB
Pilot scores: {'full_causal': 0.47694607607332845, 'full_bidirectional': 0.405406948263082} | promoted: full_causal


## 7. Optional controlled improvements and compulsory methods

In [12]:
extra=[dict(selected_run,name='partial',method='partial',lr=3e-5),
       dict(selected_run,name='lorma_plus',method='lorma',lr=2e-4)]
for run in extra:
    launch(train_worker,run,False); CANDIDATES.append(str(OUT/run['name']/'best.pt'))
eligible=[p for p in CANDIDATES if not json.loads((Path(p).parent/'run.json').read_text())['student']]
teacher=max(eligible,key=lambda p:json.loads((Path(p).parent/'best_metrics.json').read_text())['macro_f1'])
for kd in [False,True]:
    run=dict(selected_run,name='student_kd' if kd else 'student_gold',student=True,
             teacher=teacher if kd else None,method='full',lr=3e-5)
    launch(train_worker,run,False); CANDIDATES.append(str(OUT/run['name']/'best.pt'))

partial: 261,590,457 parameters | 76,748,217 trainable | 747 windows
partial epoch 1 | loss 2.0308 | macro-F1 0.1474 | 0.7 min | 2.33 GiB


/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


partial epoch 2 | loss 0.1316 | macro-F1 0.4297 | 0.7 min | 2.33 GiB
partial epoch 3 | loss 0.0729 | macro-F1 0.4531 | 0.7 min | 2.33 GiB
partial epoch 4 | loss 0.0491 | macro-F1 0.4829 | 0.7 min | 2.33 GiB
partial epoch 5 | loss 0.0318 | macro-F1 0.4963 | 0.7 min | 2.33 GiB
partial epoch 6 | loss 0.0196 | macro-F1 0.4427 | 0.7 min | 2.33 GiB
partial epoch 7 | loss 0.0120 | macro-F1 0.4826 | 0.7 min | 2.33 GiB
partial epoch 8 | loss 0.0077 | macro-F1 0.4987 | 0.7 min | 2.33 GiB
partial epoch 9 | loss 0.0056 | macro-F1 0.4689 | 0.7 min | 2.33 GiB
partial epoch 10 | loss 0.0044 | macro-F1 0.4871 | 0.7 min | 2.33 GiB
partial epoch 11 | loss 0.0039 | macro-F1 0.4967 | 0.7 min | 2.33 GiB
lorma_plus: 264,809,913 parameters | 3,274,233 trainable | 747 windows
lorma_plus epoch 1 | loss 2.0378 | macro-F1 0.1951 | 1.7 min | 1.19 GiB


/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


lorma_plus epoch 2 | loss 0.1404 | macro-F1 0.3730 | 1.7 min | 1.19 GiB
lorma_plus epoch 3 | loss 0.0867 | macro-F1 0.4844 | 1.7 min | 1.19 GiB
lorma_plus epoch 4 | loss 0.0639 | macro-F1 0.5052 | 1.7 min | 1.19 GiB
lorma_plus epoch 5 | loss 0.0489 | macro-F1 0.5108 | 1.7 min | 1.19 GiB
lorma_plus epoch 6 | loss 0.0335 | macro-F1 0.4679 | 1.7 min | 1.19 GiB
lorma_plus epoch 7 | loss 0.0219 | macro-F1 0.4992 | 1.7 min | 1.19 GiB
lorma_plus epoch 8 | loss 0.0141 | macro-F1 0.4917 | 1.6 min | 1.19 GiB
student_gold: 146,551,737 parameters | 146,551,737 trainable | 747 windows
student_gold epoch 1 | loss 1.4819 | macro-F1 0.1716 | 0.8 min | 3.03 GiB


/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


student_gold epoch 2 | loss 0.1311 | macro-F1 0.3783 | 0.8 min | 3.03 GiB
student_gold epoch 3 | loss 0.0511 | macro-F1 0.4120 | 0.8 min | 3.03 GiB
student_gold epoch 4 | loss 0.0215 | macro-F1 0.3763 | 0.8 min | 3.03 GiB
student_gold epoch 5 | loss 0.0114 | macro-F1 0.3849 | 0.8 min | 3.03 GiB
student_gold epoch 6 | loss 0.0045 | macro-F1 0.3971 | 0.8 min | 3.03 GiB
student_kd: 146,551,737 parameters | 146,551,737 trainable | 747 windows
student_kd epoch 1 | loss 2.9457 | macro-F1 0.2377 | 1.2 min | 4.01 GiB


/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


student_kd epoch 2 | loss 0.3001 | macro-F1 0.4142 | 1.2 min | 4.01 GiB
student_kd epoch 3 | loss 0.1480 | macro-F1 0.3977 | 1.2 min | 4.01 GiB
student_kd epoch 4 | loss 0.1083 | macro-F1 0.4257 | 1.1 min | 4.01 GiB
student_kd epoch 5 | loss 0.0864 | macro-F1 0.4082 | 1.2 min | 4.01 GiB
student_kd epoch 6 | loss 0.0701 | macro-F1 0.4249 | 1.1 min | 4.01 GiB
student_kd epoch 7 | loss 0.0599 | macro-F1 0.4408 | 1.1 min | 4.01 GiB
student_kd epoch 8 | loss 0.0518 | macro-F1 0.4344 | 1.1 min | 4.01 GiB
student_kd epoch 9 | loss 0.0456 | macro-F1 0.4389 | 1.1 min | 4.01 GiB
student_kd epoch 10 | loss 0.0424 | macro-F1 0.4180 | 1.1 min | 4.01 GiB


## 8. Cache held-out logits once; compare decoding without rerunning the backbone

In [13]:
def cache_worker(rank,path,ids,destination):
    torch.set_num_threads(2); torch.cuda.set_device(rank); device=torch.device('cuda',rank)
    ck=torch.load(path,map_location='cpu',weights_only=False); run=ck['run']
    model=make_model(ck['config'],run,phase1=False); model.load_state_dict(ck['model']); model.to(device).eval()
    with open(run['prepared'],'rb') as f: pack=cloudpickle.load(f)
    if ck['manifest'] != pack['manifest']: raise ValueError('Inference inputs differ from the checkpoint.')
    records=pack['records']
    arrays={}; torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize(); start=time.monotonic()
    for k in ids[rank::2]:
        for j,z in enumerate(infer_doc(model,records[k],run,device)): arrays[k+'__'+str(j)]=z.astype(np.float16)
    torch.cuda.synchronize()
    np.savez_compressed(f'{destination}.{rank}.npz',**arrays)
    Path(f'{destination}.{rank}.json').write_text(json.dumps(dict(seconds=time.monotonic()-start,
        peak_gib=torch.cuda.max_memory_allocated()/2**30,documents=len(ids[rank::2]))))

comparison=[]; selected_score=-1.; selected=None
for path in CANDIDATES:
    dest=Path(path).parent/'valid_logits'
    launch(cache_worker,path,valid_ids,str(dest))
    cached={}
    for rank in range(2):
        with np.load(f'{dest}.{rank}.npz') as z: cached.update({k:z[k].astype(np.float32) for k in z.files})
    for refine in [False,True]:
        for bias in [-.4,0.,.4]:
            pred={k:decode(records[k],[cached[k+'__'+str(j)] for j in range(3)],refine,bias) for k in valid_ids}
            metrics=score_ner(pred,gold,valid_ids)
            entry=dict(checkpoint=path,refine=refine,bias=bias,**metrics); comparison.append(entry)
            if metrics['macro_f1']>selected_score:
                selected_score=metrics['macro_f1']; selected=entry; selected_pred=pred
(OUT/'comparison.json').write_text(json.dumps(comparison,indent=2))
(OUT/'selection.json').write_text(json.dumps(selected,indent=2))
(OUT/'selected_valid_predictions.json').write_text(json.dumps(selected_pred))
print(f"Selected: {Path(selected['checkpoint']).parent.name} | refined={selected['refine']} | bias={selected['bias']} | strict macro-F1={selected_score:.4f}")
for label,row in selected['per_class'].items(): print(label,round(row['f1'],4),'support',row['support'])

Selected: lorma_plus | refined=True | bias=0.4 | strict macro-F1=0.5264
APP 0.2302 support 81
RESP 0.2787 support 73
A.COUNSEL 0.4828 support 30
R.COUNSEL 0.6389 support 40
JUDGE 0.5286 support 57
WIT 0.4746 support 35
AUTH 0.359 support 198
COURT 0.7924 support 223
STAT 0.6709 support 228
PREC 0.4393 support 241
DATE 0.9082 support 305
CASENO 0.5134 support 187


In [14]:
errors=Counter(); examples=[]
for k in valid_ids:
    p=set(map(tuple,selected_pred[k])); g={tuple(e[:3]) for e in gold[k]}
    for span in p-g:
        s,e,l=span
        kind=('wrong_label' if any((s,e)==x[:2] for x in g) else
              'boundary' if any(l==x[2] and s<x[1] and e>x[0] for x in g) else 'false_positive')
        errors[kind]+=1
        if len(examples)<40: examples.append(dict(id=k,type=kind,span=span,text=texts[k][s:e]))
    errors['missed_gold']+=len(g-p)
(OUT/'errors.json').write_text(json.dumps(dict(counts=dict(errors),examples=examples),indent=2))
print('Validation errors:',dict(errors))

rng=np.random.default_rng(1781); boot=[]
for _ in range(500):
    sampled=rng.choice(valid_ids,len(valid_ids),replace=True).tolist()
    boot.append(score_ner(selected_pred,gold,sampled)['macro_f1'])
interval=np.quantile(boot,[.025,.975]).tolist()
(OUT/'validation_uncertainty.json').write_text(json.dumps(dict(document_bootstrap_95=interval,
    note='Conditional on model selection on the same 14 documents; not an unbiased generalization interval.')))
print('Document-bootstrap interval:',[round(x,4) for x in interval])

Validation errors: {'boundary': 288, 'false_positive': 302, 'wrong_label': 44, 'missed_gold': 697}
Document-bootstrap interval: [0.4454, 0.6005]


## 9. Optional final refit and official submission

In [16]:
from pathlib import Path
import shutil

OUT = Path('/kaggle/working/phase2_arch4')

for p in OUT.glob('*/last.pt'):
    p.unlink()
for p in OUT.glob('*/*.tmp'):
    p.unlink()
for p in OUT.glob('*/valid_logits.*.npz'):
    p.unlink()
(OUT / 'worker.pkl').unlink(missing_ok=True)

usage = shutil.disk_usage('/kaggle/working')
print(f'Free disk space: {usage.free / 2**30:.1f} GiB')

Free disk space: 13.5 GiB


In [17]:
REFIT_ALL=True
ck=torch.load(selected['checkpoint'],map_location='cpu',weights_only=False)
run=dict(ck['run'],name='refit_all',refit=True,epochs=ck['epoch'],stop_epoch=ck['epoch'])
del ck
launch(train_worker,run,False)
submission_checkpoint=str(OUT/'refit_all'/'best.pt')
dest=OUT/'test_logits'
launch(cache_worker,submission_checkpoint,list(test),str(dest))
cached={}
for rank in range(2):
    with np.load(f'{dest}.{rank}.npz') as z: cached.update({k:z[k].astype(np.float32) for k in z.files})
prediction={k:decode(records[k],[cached[k+'__'+str(j)] for j in range(3)],selected['refine'],selected['bias']) for k in test}
(OUT/'prediction.json').write_text(json.dumps(prediction,ensure_ascii=False))
with zipfile.ZipFile(OUT/'submission.zip','w',zipfile.ZIP_DEFLATED) as z:
    z.write(OUT/'prediction.json','prediction.json')
(OUT/'submission_manifest.json').write_text(json.dumps(dict(checkpoint=submission_checkpoint,
    refine=selected['refine'],bias=selected['bias'],refit=REFIT_ALL,documents=len(test)),indent=2))
print(f'{len(prediction)} documents | {sum(map(len,prediction.values()))} entities | {OUT / "submission.zip"}')

refit_all: 264,809,913 parameters | 3,274,233 trainable | 901 windows
refit_all epoch 1 | loss 1.3031 | macro-F1 0.0000 | 2.0 min | 1.19 GiB


/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


refit_all epoch 2 | loss 0.1148 | macro-F1 0.0000 | 2.0 min | 1.19 GiB
refit_all epoch 3 | loss 0.0810 | macro-F1 0.0000 | 2.0 min | 1.19 GiB
refit_all epoch 4 | loss 0.0598 | macro-F1 0.0000 | 2.0 min | 1.19 GiB
refit_all epoch 5 | loss 0.0447 | macro-F1 0.0000 | 2.0 min | 1.19 GiB
35 documents | 3985 entities | /kaggle/working/phase2_arch4/submission.zip


In [1]:
import json
from pathlib import Path

OUT = Path('/kaggle/working/phase2_arch4')
submission = json.loads((OUT / 'submission_manifest.json').read_text())
checkpoint = Path(submission['checkpoint'])
run = json.loads((checkpoint.parent / 'run.json').read_text())

result = {
    'test_documents': 35,
    'strict_macro_f1': 0.495445,
    'strict_macro_precision': 0.529517,
    'strict_macro_recall': 0.482692,
    'checkpoint': str(checkpoint),
    'method': run['method'],
    'causal_attention': run['causal'],
    'selection': json.loads((OUT / 'selection.json').read_text()),
    'submission': submission
}
(OUT / 'phase2_result.json').write_text(json.dumps(result, indent=2))
print(OUT / 'phase2_result.json')

/kaggle/working/phase2_arch4/phase2_result.json


In [2]:
import json, shutil
from pathlib import Path

OUT = Path('/kaggle/working/phase2_arch4')
submission = json.loads((OUT / 'submission_manifest.json').read_text())
checkpoint = Path(submission['checkpoint'])
run = json.loads((checkpoint.parent / 'run.json').read_text())
selection = json.loads((OUT / 'selection.json').read_text())

for name in ['config.json', 'tokenizer.json']:
    source = Path(run['base']) / name
    if source.resolve() != (OUT / name).resolve():
        shutil.copy2(source, OUT / name)
shutil.copy2(Path(DATA_DIR) / 'labels.txt', OUT / 'labels.txt')

result = {
    'phase': 2,
    'test_documents': 35,
    'strict_macro_f1': 0.495445,
    'strict_macro_precision': 0.529517,
    'strict_macro_recall': 0.482692,
    'phase2_checkpoint': str(checkpoint.relative_to('/kaggle/working')),
    'method': run['method'],
    'causal_attention': run['causal'],
    'selection': selection,
    'submission': submission,
    'phase1_checkpoint_source': str(Path(run['base']) / 'best.pt'),
    'important_files': [
        'phase2_arch4/phase2_result.json',
        'phase2_arch4/prediction.json',
        'phase2_arch4/submission.zip',
        'phase2_arch4/selection.json',
        'phase2_arch4/comparison.json',
        'phase2_arch4/split.json',
        'phase2_arch4/manifest.json',
        'phase2_arch4/labels.txt',
        'phase2_arch4/config.json',
        'phase2_arch4/tokenizer.json'
    ]
}
(OUT / 'phase2_result.json').write_text(json.dumps(result, indent=2))

removed = 0
for pattern in ['*/last.pt', '*/valid_logits.*.npz', 'test_logits.*.npz', '*.tmp', '*/*.tmp']:
    for path in OUT.glob(pattern):
        removed += path.stat().st_size
        path.unlink()

print(f"Saved result record: {OUT / 'phase2_result.json'}")
print(f"Freed {removed / 2**30:.2f} GiB; kept the best checkpoints and experiment metrics.")

Saved result record: /kaggle/working/phase2_arch4/phase2_result.json
Freed 1.03 GiB; kept the best checkpoints and experiment metrics.


## Interpretation, constraints and next experiments